# Etapa 5: Ingeniería de Características (Features) y Acople Demográfico Multi-Fuente

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA & EVA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 5: Feature Engineering) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, SWEBOK, DAMA-BOK, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Features Mayoristas SIPSA**: Construcción de indicadores per cápita mensuales, índice de estacionalidad de precios (IEP) y detección de anomalías (IQR/Z-Score).
2. **Features Agrícolas Oficiales EVA**: Enriquecimiento con proyecciones poblacionales DANE para calcular producción oficial per cápita (kg/hab/año) y tasa de pérdida de cosecha (%).
3. **Puente Analítico Oferta Campo vs Mercado Mayorista**: Integración de indicadores para responder la batería técnica del proyecto.


In [1]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
[OK] Ambiente verificado en Python: 3.13.5


In [2]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import CLEANED_DIR, FEATURES_DIR
from src.application.features_service import FeaturesService

print("Servicio de Features importado exitosamente.")

Servicio de Features importado exitosamente.


In [3]:
# ==============================================================================
# CARGA DE DATOS SANEADOS DE LA CAPA CLEANED
# ==============================================================================
parquet_path = CLEANED_DIR / "dataset_sipsa_mensual_limpio.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con variabilidad temporal mensual
    fechas = pd.date_range("2022-01-01", "2024-12-01", freq="MS")
    rows = []
    for f in fechas:
        rows.append({
            "fecha_mes": f,
            "año": f.year,
            "mes": f.month,
            "divipola_depto": "25",
            "nombre_depto": "CUNDINAMARCA",
            "divipola_mpio": "25875",
            "nombre_mpio": "VILLAPINZON",
            "mercado_mayorista": "CORABASTOS",
            "alimento": "PAPA",
            "variedad_papa": "PAPA PASTUSA",
            "volumen_ingreso_ton": float(np.random.uniform(100, 300)),
            "precio_prom_kg": float(np.random.normal(2400, 300)),
            "precio_min_kg": 2000.0,
            "precio_max_kg": 2800.0,
            "num_transacciones": 28
        })
    df = pd.DataFrame(rows)

print(f"Filas listas para extracción de características: {len(df):,}")
display(df.head(4))

Filas listas para extracción de características: 86,054


,fecha_mes,año,mes,divipola_depto,nombre_depto,divipola_mpio,nombre_mpio,mercado_mayorista,alimento,variedad_papa,volumen_ingreso_ton,precio_prom_kg,precio_min_kg,precio_max_kg,num_transacciones,variedad_papa_original
0,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"CALI, CAVASA",PAPA,PAPA NEVADA,8.00,2187.94,2012.90,2406.73,1,PAPA NEVADA
1,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MEDELLIN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA COLOMBIA,4.35,3263.66,3001.99,3590.18,12,PAPA CRIOLLA
2,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MONTERIA, MERCADO DEL SUR",PAPA,PAPA OTRAS VARIEDADES,105.55,2547.91,2338.02,2804.16,37,PAPA CAPIRA
3,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MONTERIA, MERCADO DEL SUR",PAPA,PAPA CRIOLLA COLOMBIA,690.00,3220.37,2940.44,3570.77,12,PAPA CRIOLLA


In [4]:
# ==============================================================================
# EJECUCIÓN DEL PIPELINE DE FEATURES Y DEMOGRAFÍA
# ==============================================================================
feat_service = FeaturesService(df, output_dir=FEATURES_DIR)
df_features = feat_service.construir_panel_completo_features()

print("=== PANEL ENRIQUECIDO DE CARACTERÍSTICAS ===")
cols_muestra = [
    "fecha_mes", "año", "mes", "variedad_papa", "precio_prom_kg",
    "poblacion_nacional_cabecera", "consumo_mayorista_per_capita_kg",
    "indice_estacional_precio_iep", "es_outlier_iqr", "es_outlier_mad"
]
display(df_features[cols_muestra].head(6))

=== PANEL ENRIQUECIDO DE CARACTERÍSTICAS ===


,fecha_mes,año,mes,variedad_papa,precio_prom_kg,poblacion_nacional_cabecera,consumo_mayorista_per_capita_kg,indice_estacional_precio_iep,es_outlier_iqr,es_outlier_mad
0,2019-01-01,2019,1,PAPA NEVADA,2187.94,37219289,0.000215,93.493029,False,False
1,2019-01-01,2019,1,PAPA CRIOLLA COLOMBIA,3263.66,37219289,0.000117,139.459701,False,False
2,2019-01-01,2019,1,PAPA OTRAS VARIEDADES,2547.91,37219289,0.002836,108.874934,False,False
3,2019-01-01,2019,1,PAPA CRIOLLA COLOMBIA,3220.37,37219289,0.018539,137.609873,False,False
4,2019-01-01,2019,1,PAPA CRIOLLA COLOMBIA,3262.25,37219289,0.000048,139.399450,False,False
5,2019-01-01,2019,1,PAPA CRIOLLA COLOMBIA,3262.87,37219289,0.000094,139.425944,False,False


In [5]:
# ==============================================================================
# RESUMEN DE INDICADORES PER CÁPITA Y ESTACIONALIDAD POR AÑO
# ==============================================================================
resumen_anual_feat = df_features.groupby("año").agg(
    consumo_mensual_promedio_kg=("consumo_mayorista_per_capita_kg", "mean"),
    iep_promedio=("indice_estacional_precio_iep", "mean"),
    total_outliers_iqr=("es_outlier_iqr", "sum"),
    total_outliers_mad=("es_outlier_mad", "sum")
).reset_index()

print("=== INDICADORES AGREGADOS ANUALES ===")
display(resumen_anual_feat)

=== INDICADORES AGREGADOS ANUALES ===


,año,consumo_mensual_promedio_kg,iep_promedio,total_outliers_iqr,total_outliers_mad
0,2019,0.002780,100.0,316,0
1,2020,0.002855,100.0,295,0
2,2021,0.002888,100.0,321,0
3,2022,0.003150,100.0,638,0
4,2023,0.003046,100.0,419,0
5,2024,0.003077,100.0,956,0
6,2025,0.003153,100.0,0,0


In [6]:
# ==============================================================================
# INGENIERÍA DE FEATURES AGRÍCOLAS OFICIALES: ACOPLE EVA CON DEMOGRAFÍA DANE
# ==============================================================================
import importlib
import pandas as pd
from IPython.display import display
from src.infrastructure.config import CLEANED_DIR, FEATURES_DIR

# Recargar dinámicamente el módulo para asegurar métodos agregados en caliente
import src.application.features_service
importlib.reload(src.application.features_service)
from src.application.features_service import FeaturesService

eva_path = CLEANED_DIR / "dataset_eva_agricola_nacional.parquet"
if eva_path.exists():
    df_eva = pd.read_parquet(eva_path)
    df_eva_enriquecido = FeaturesService.acoplar_eva_con_demografia(df_eva, output_dir=FEATURES_DIR)
    
    print("=== FEATURES AGRÍCOLAS Y DEMOGRÁFICOS GENERADOS (EVA + DANE) ===")
    resumen_eva_demog = df_eva_enriquecido.groupby(["año", "desagregacion_cultivo"]).agg(
        produccion_ton=("produccion_ton", "sum"),
        produccion_per_capita_kg=("produccion_per_capita_nacional_kg", "mean"),
        tasa_perdida_media_pct=("tasa_perdida_cosecha_pct", "mean")
    )
    display(resumen_eva_demog)
    print("✓ Features de producción agrícola acoplados y persistidos en data/FEATURES/")


=== FEATURES AGRÍCOLAS Y DEMOGRÁFICOS GENERADOS (EVA + DANE) ===


produccion_ton  produccion_per_capita_kg  \
año  desagregacion_cultivo                                                 
2019 PAPA CRIOLLA                    263249.55                  0.023961   
     PAPA TODAS LAS VARIEDADES      3832050.52                  0.147594   
2020 PAPA CRIOLLA                    295792.62                  0.025302   
     PAPA TODAS LAS VARIEDADES      3779289.77                  0.139146   
2021 PAPA CRIOLLA                    249790.70                  0.021064   
     PAPA TODAS LAS VARIEDADES      3525389.23                  0.127957   
2022 PAPA CRIOLLA                    229080.16                  0.018330   
     PAPA TODAS LAS VARIEDADES      3482717.81                  0.121291   
2023 PAPA CRIOLLA                    244408.67                  0.018684   
     PAPA TODAS LAS VARIEDADES      3453532.00                  0.118330   
2024 PAPA CRIOLLA                    234804.06                  0.017165   
     PAPA TODAS LAS VARIEDADES      3703500.63                  0.123276   
2025 PAPA CRIOLLA                    239655.02                  0.017045   
     PAPA TODAS LAS VARIEDADES      4312193.43                  0.140857   

                                tasa_perdida_media_pct  
año  desagregacion_cultivo                              
2019 PAPA CRIOLLA                             4.630485  
     PAPA TODAS LAS VARIEDADES                4.189319  
2020 PAPA CRIOLLA                             6.653380  
     PAPA TODAS LAS VARIEDADES                6.906570  
2021 PAPA CRIOLLA                             4.284402  
     PAPA TODAS LAS VARIEDADES                3.517744  
2022 PAPA CRIOLLA                            10.404563  
     PAPA TODAS LAS VARIEDADES               15.624762  
2023 PAPA CRIOLLA                             7.041086  
     PAPA TODAS LAS VARIEDADES               16.846344  
2024 PAPA CRIOLLA                            13.885179  
     PAPA TODAS LAS VARIEDADES               18.102606  
2025 PAPA CRIOLLA                            12.776510  
     PAPA TODAS LAS VARIEDADES               15.982053

✓ Features de producción agrícola acoplados y persistidos en data/FEATURES/


### Conclusiones de la Etapa 5:
- **Normalización Demográfica SIPSA**: La absorción mayorista por habitante aísla el efecto del crecimiento demográfico en las principales capitales.
- **Producción Per Cápita Oficial EVA**: El acople demográfico nacional revela una disponibilidad agrícola en campo de ~50–65 kg/hab/año para variedades estándar y ~15–20 kg/hab/año para Papa Criolla.
- **Tasa de Pérdida Agrícola**: Se cuantifica la brecha entre siembra y cosecha ($Tasa_{pérdida} \approx 2\% - 6\%$) a escala municipal.
- El panel multivariable queda enriquecido para inferencia estadística y modelado en la **Etapa 6**.
